# Week 36, Day 1: State

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w036d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
TRANSITIONS = {
    ("locked", "coin"): "unlocked",
    ("unlocked", "push"): "locked",
}

state = "locked"
for event in ["push", "coin", "coin", "push", "push"]:
    new = TRANSITIONS.get((state, event), state)
    print(f"{state:8} + {event:4} -> {new}")
    state = new

In [ ]:
class Locked:
    name = "locked"
    def coin(self, gate):
        gate.state = Unlocked()
        return "unlocked: walk through"
    def push(self, gate):
        return "locked: insert a coin"

class Unlocked:
    name = "unlocked"
    def coin(self, gate):
        gate.refunds += 1
        return "already open: coin returned"
    def push(self, gate):
        gate.state = Locked()
        gate.passed += 1
        return "thank you"

class Turnstile:
    def __init__(self):
        self.state = Locked()
        self.passed = 0
        self.refunds = 0

    def coin(self):
        return self.state.coin(self)

    def push(self):
        return self.state.push(self)

t = Turnstile()
for event in ["push", "coin", "coin", "push", "push"]:
    print(f"{event}: {getattr(t, event)()}  (now {t.state.name})")
print(t.passed, t.refunds)

In [ ]:
from enum import Enum

class Status(Enum):
    PLACED = "placed"
    PAID = "paid"
    SHIPPED = "shipped"
    DELIVERED = "delivered"
    CANCELLED = "cancelled"

MOVES = {
    (Status.PLACED, "pay"): Status.PAID,
    (Status.PLACED, "cancel"): Status.CANCELLED,
    (Status.PAID, "ship"): Status.SHIPPED,
    (Status.PAID, "cancel"): Status.CANCELLED,
    (Status.SHIPPED, "deliver"): Status.DELIVERED,
}

class Order:
    def __init__(self, code):
        self.code = code
        self.status = Status.PLACED
        self.history = [Status.PLACED]

    def handle(self, event):
        key = (self.status, event)
        if key not in MOVES:
            allowed = sorted(e for (s, e) in MOVES if s is self.status)
            raise ValueError(f"can't {event} a {self.status.value} order; allowed: {allowed}")
        self.status = MOVES[key]
        self.history.append(self.status)

o = Order("A17")
for event in ["pay", "ship", "cancel"]:
    try:
        o.handle(event)
    except ValueError as err:
        print("ValueError:", err)
print([s.value for s in o.history])

> **Key idea.** State (Gang of Four): when an object's behaviour depends on its state, give each state its own object with a method per event; the context passes events to its current state, which acts and chooses the next state, replacing `if status == ...` chains in every method. A transition table, a dictionary from `(state, event)` to the next state (often with Enum states), is simpler when states differ mainly in what's allowed; it's data you can print and test. Reject events that aren't allowed with a clear error listing what is, and let the state change only through the machine.

### Exercise: A table for a traffic light

A pedestrian crossing's light has states `"green"`, `"amber"`, `"red"` and `"red_amber"` for the traffic. Write a transition table `LIGHT` and a class `Crossing` (starting at `"green"`) with `handle(event)`, which moves to the next state and returns it. The events: from green, `"button"` goes to amber; from amber, `"timer"` goes to red; from red, `"timer"` goes to red_amber; from red_amber, `"timer"` goes to green. Any other event in a state raises `ValueError`. `Crossing` must not use `if`/`elif` chains on the state.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_light():\n    \"the crossing\"\n    C = ns[\"Crossing\"]\n    c = C()\n    assert [c.handle(e) for e in [\"button\", \"timer\", \"timer\", \"timer\"]] == [\"amber\", \"red\", \"red_amber\", \"green\"]\n    assert c.state == \"green\" and len(LIGHT) == 4\n    for e in [\"timer\", \"honk\"]:\n        try:\n            c.handle(e)\n            assert False, f\"{e} on green should raise ValueError\"\n        except ValueError:\n            pass\n    assert c.state == \"green\" and \"elif\" not in source")

### Exercise: A vending machine's states

Write a vending machine with the State pattern. `Machine(price)` has `insert(pence)`, `choose()` and `refund()`, each returning a message, and `credit` and `dispensed` attributes. It passes each event to its state object (`Idle`, `HasCredit` or `SoldOut`):

- `Idle`: `insert` adds to credit and moves to `HasCredit` (`"credit 50"`); `choose` gives `"insert money first"`; `refund` gives `"nothing to refund"`.
- `HasCredit`: `insert` adds to credit (`"credit 100"`); `choose` dispenses if credit is at least the price (`"enjoy! change 20"`, with credit back to 0, `dispensed` up by one, and the machine moving to `Idle`), otherwise `"need 30 more"`; `refund` returns the credit (`"refunded 50"`) and moves to `Idle`.
- `SoldOut` (when `Machine(price, stock=0)` is made, or after the last item is dispensed): every event gives `"sold out"`, except that `refund` with credit refunds it.

`Machine` itself must not check which state it's in.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_machine():\n    \"states\"\n    M = ns[\"Machine\"]\n    m = M(120, stock=2)\n    assert m.choose() == \"insert money first\" and m.refund() == \"nothing to refund\"\n    assert m.insert(50) == \"credit 50\" and m.insert(50) == \"credit 100\" and m.choose() == \"need 20 more\"\n    assert m.insert(40) == \"credit 140\" and m.choose() == \"enjoy! change 20\" and m.credit == 0 and m.dispensed == 1\n    assert m.insert(100) == \"credit 100\" and m.refund() == \"refunded 100\" and m.credit == 0\n    assert m.insert(200) == \"credit 200\" and m.choose() == \"enjoy! change 80\" and type(m.state).__name__ == \"SoldOut\"\n    assert m.insert(50) == \"sold out\" and m.choose() == \"sold out\" and m.refund() == \"nothing to refund\"\n    empty = M(100, stock=0)\n    assert empty.insert(100) == \"sold out\" and empty.credit == 0\n    import ast\n    cls = next(n for n in ast.walk(ast.parse(source)) if isinstance(n, ast.ClassDef) and n.name == \"Machine\")\n    assert not any(isinstance(n, ast.If) for n in ast.walk(cls)), \"Machine shouldn't check its state with if\"")

### Exercise: Checking a table

Because a transition table is data, it can be checked. Write `allowed(table, state)` (the events allowed in a state, sorted), `reachable(table, start)` (every state reachable from `start` by any sequence of events, including `start`, as a set) and `dead_ends(table, start, finals)` (the reachable states that aren't in `finals` and have no events at all, sorted). States and events are strings.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checks():\n    \"checking a machine\"\n    T = {(\"draft\", \"submit\"): \"review\", (\"review\", \"approve\"): \"published\", (\"review\", \"reject\"): \"draft\",\n         (\"published\", \"archive\"): \"archived\", (\"review\", \"lose\"): \"limbo\", (\"ghost\", \"x\"): \"draft\"}\n    assert allowed(T, \"review\") == [\"approve\", \"lose\", \"reject\"] and allowed(T, \"archived\") == []\n    assert reachable(T, \"draft\") == {\"draft\", \"review\", \"published\", \"archived\", \"limbo\"} and reachable(T, \"archived\") == {\"archived\"}\n    assert dead_ends(T, \"draft\", {\"archived\"}) == [\"limbo\"] and dead_ends(T, \"draft\", {\"archived\", \"limbo\"}) == []")

### Exercise: Transitions with actions

Write a class `Ticket` for a help-desk ticket, whose transition table `FLOW` maps `(state, event)` to `(next_state, action)`, where an action is a function taking the ticket. The states are `"open"`, `"assigned"`, `"resolved"` and `"closed"`. Events: from open, `"assign"` (to assigned; the action records the time of the change: increment `ticket.changes` and append `"assigned"` to `ticket.log`); from assigned, `"resolve"` (to resolved; log `"resolved"`) and `"unassign"` (back to open; log `"unassigned"`); from resolved, `"reopen"` (to assigned; log `"reopened"`) and `"close"` (to closed; log `"closed"`). Every action also adds 1 to `ticket.changes`. `Ticket.handle(event)` raises `ValueError` for events not in the table, and otherwise changes the state and runs the action.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ticket():\n    \"actions\"\n    T = ns[\"Ticket\"]\n    t = T()\n    for e in [\"assign\", \"resolve\", \"reopen\", \"resolve\", \"close\"]:\n        t.handle(e)\n    assert t.state == \"closed\" and t.log == [\"assigned\", \"resolved\", \"reopened\", \"resolved\", \"closed\"] and t.changes == 5\n    for e in [\"reopen\", \"assign\"]:\n        try:\n            t.handle(e)\n            assert False, f\"{e} on a closed ticket should raise ValueError\"\n        except ValueError:\n            pass\n    assert t.changes == 5 and all(callable(a) for _, a in FLOW.values()) and len(FLOW) == 5")

### Bug hunt: A state that never changes

After a coin, the turnstile should be unlocked, so a push lets someone through, but `Locked.coin` forgets to change the state: this prints `['locked: insert a coin', 'unlocked: walk through', 'locked: insert a coin']`. Fix it so it prints `['locked: insert a coin', 'unlocked: walk through', 'thank you']`.

In [ ]:
class Locked:
    def coin(self, gate):
        return "unlocked: walk through"
    def push(self, gate):
        return "locked: insert a coin"

class Unlocked:
    def coin(self, gate):
        return "already open"
    def push(self, gate):
        gate.state = Locked()
        return "thank you"

class Turnstile:
    def __init__(self):
        self.state = Locked()

t = Turnstile()
print([t.state.push(t), t.state.coin(t), t.state.push(t)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unlocked after a coin\"\n    expect_output(\"['locked: insert a coin', 'unlocked: walk through', 'thank you']\")")

### Bug hunt: Going round the machine

Code elsewhere sets the order's status directly, skipping the rules, so a cancelled order ends up shipped: this prints `shipped`. The rule is that the state only changes through `handle`. Fix the last part of the program to use `handle`, so it prints `can't ship a cancelled order` and then `cancelled`.

In [ ]:
MOVES = {("placed", "pay"): "paid", ("placed", "cancel"): "cancelled", ("paid", "ship"): "shipped"}

class Order:
    def __init__(self):
        self.status = "placed"

    def handle(self, event):
        if (self.status, event) not in MOVES:
            raise ValueError(f"can't {event} a {self.status} order")
        self.status = MOVES[(self.status, event)]

o = Order()
o.handle("cancel")
o.status = "shipped"
print(o.status)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"through the machine\"\n    expect_output(\"can't ship a cancelled order\\ncancelled\")\n    assert \".status =\" not in source.split(\"o.handle(\\\"cancel\\\")\")[1], \"don't set the status directly\"")

### Bug hunt: An event that silently does nothing

`handle` uses `.get` with the current state as the default, so an event that isn't allowed is silently ignored, and a typo like `"shp"` goes unnoticed: this prints `paid`. Make unknown events raise `ValueError`, so the program prints `ValueError` and then `paid`.

In [ ]:
MOVES = {("placed", "pay"): "paid", ("paid", "ship"): "shipped"}

class Order:
    def __init__(self):
        self.status = "placed"

    def handle(self, event):
        self.status = MOVES.get((self.status, event), self.status)

o = Order()
o.handle("pay")
try:
    o.handle("shp")
except ValueError:
    print("ValueError")
print(o.status)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no silent failures\"\n    expect_output(\"ValueError\\npaid\")")

### Puzzle: A context passing events on

Arrange the lines so that a `Door` passes each event to its current state object. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
T = {("a", "x"): "b", ("b", "x"): "c", ("b", "y"): "a", ("c", "y"): "a"}

def run(events, state="a"):
    for e in events:
        state = T.get((state, e), state)
    return state

p = ___   # run(["x"])
q = ___   # run(["x", "x"])
r = ___   # run(["x", "y", "x", "x", "y"])
s = ___   # run(["y", "y"])
t = ___   # sorted(e for (st, e) in T if st == "b")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"b\", \"c\", \"a\"), \"p to r\"\n    assert s == \"a\" and t == [\"x\", \"y\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a document workflow

A newsroom's articles go through a workflow, and who may do what depends on the state. Write it with a transition table that includes **permissions**:

- `Role`, an Enum: `WRITER`, `EDITOR`, `LEGAL`.
- `WORKFLOW`, a dictionary from `(state, event)` to `(next_state, roles)`, where `roles` is a set of the roles allowed to do it:
  - `draft` + `submit` -> `review` (writer)
  - `review` + `request_changes` -> `draft` (editor)
  - `review` + `approve` -> `legal_check` (editor)
  - `legal_check` + `clear` -> `scheduled` (legal)
  - `legal_check` + `block` -> `draft` (legal)
  - `scheduled` + `publish` -> `published` (editor)
  - `scheduled` + `pull` -> `review` (editor, legal)
- `Article(title)`, starting in `draft`, with `handle(event, role)`: an event not allowed in the current state raises `ValueError` (message listing the allowed events, sorted); a role that isn't allowed raises `PermissionError`; otherwise the state changes and `(state, event, role)` is recorded in `history` (as the **new** state, the event and the `Role`). `next_steps(role)` returns the events that role may do right now, sorted.
- `path_exists(start, end)`: whether `end` can be reached from `start` through `WORKFLOW`.

In [ ]:
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_workflow():\n    \"the workflow\"\n    A, R = ns[\"Article\"], ns[\"Role\"]\n    a = A(\"Rail strike called off\")\n    a.handle(\"submit\", R.WRITER)\n    for event, role, err in [(\"approve\", R.WRITER, PermissionError), (\"publish\", R.EDITOR, ValueError), (\"clear\", R.LEGAL, ValueError)]:\n        try:\n            a.handle(event, role)\n            assert False, f\"{event} by {role.name} should raise {err.__name__}\"\n        except err:\n            pass\n    assert a.next_steps(R.EDITOR) == [\"approve\", \"request_changes\"] and a.next_steps(R.WRITER) == []\n    a.handle(\"approve\", R.EDITOR)\n    a.handle(\"clear\", R.LEGAL)\n    assert a.next_steps(R.LEGAL) == [\"pull\"] and a.next_steps(R.EDITOR) == [\"publish\", \"pull\"]\n    a.handle(\"pull\", R.LEGAL)\n    a.handle(\"approve\", R.EDITOR)\n    a.handle(\"clear\", R.LEGAL)\n    a.handle(\"publish\", R.EDITOR)\n    assert a.state == \"published\" and [h[0] for h in a.history] == [\"review\", \"legal_check\", \"scheduled\", \"review\", \"legal_check\", \"scheduled\", \"published\"]\n    assert a.history[3] == (\"review\", \"pull\", R.LEGAL)\n    try:\n        a.handle(\"submit\", R.WRITER)\n        assert False\n    except ValueError as err:\n        assert \"[]\" in str(err), \"published allows no events\"\ndef test_paths():\n    \"paths\"\n    assert path_exists(\"draft\", \"published\") and path_exists(\"scheduled\", \"draft\") and not path_exists(\"published\", \"draft\") and path_exists(\"review\", \"review\")")

## Warm-up

A coin-operated turnstile can be **locked** or **unlocked**, and the same event means different things in each: a push does nothing when it's locked, and lets you through (locking it again) when it's unlocked. The whole behaviour fits in a table of `(state, event) -> next state`. That's a **state machine**, and today is about writing them cleanly.

## Behaviour that depends on state

Here's how state-dependent classes usually start:

```python
class Order:
    def pay(self):
        if self.status == "placed":
            self.status = "paid"
        elif self.status == "paid":
            raise ValueError("already paid")
        elif self.status == "cancelled":
            raise ValueError("cancelled orders can't be paid")
        ...

    def ship(self):
        if self.status == "paid":
            ...
        elif self.status == "placed":
            ...
```

Every method has the same `if` chain on the status, and every new status (`"refunded"`, `"on hold"`) means editing every method. The rules for one status are scattered across all of them, and it's easy to forget a case. The **State** pattern (Gang of Four: "allow an object to alter its behaviour when its internal state changes; the object will appear to change its class") gathers each state's rules into one place.

## The State pattern

Each state is a class with a method for each event. The object that has a state (the **context**) holds the current state object and passes each event to it; the state decides what happens and which state comes next:

- Everything about being locked is in `Locked`; everything about being unlocked is in `Unlocked`. Adding a state (say `Broken`, where every event says "out of order") is one new class.
- The context, `Turnstile`, has no `if` on its state at all; it just passes events on.
- States change the context (`gate.state = Unlocked()`), so they're passed the context as an argument.

States that hold no data of their own, like these, can be shared: one `Locked` object would do for every turnstile. When a state does need data (how many coins have been put in so far, say), it can keep it, or the context can.

## A table of transitions

When the states differ mostly in **which events are allowed and where they lead**, rather than in complicated behaviour, a **transition table** is simpler than a class per state. It's a dictionary from `(state, event)` to the next state, usually with an Enum for the states (Week 32):

The table is **data**: the whole life of an order is visible in five lines, it can be printed as documentation, checked by tests ("can every state reach the end?"), or even drawn as a diagram. The `allowed` list in the error message comes straight from it.

Tables can also hold an **action** to run on each transition, as a function: `(Status.PAID, "ship"): (Status.SHIPPED, send_tracking_email)`.

## Choosing

| Use | When |
|---|---|
| a plain Enum with one or two `if`s | a couple of states and very little behaviour depends on them |
| a transition table | the main question is "which events are allowed, and where do they lead?" |
| state classes | each state behaves quite differently, with real logic per state and event |

You'll often combine them: a table for the transitions and a small function per transition for the actions. Whichever you choose, keep two rules: **every event in every state has a defined result** (even if that result is "not allowed"), and **the state can only change through the machine**, never by code elsewhere setting `order.status` directly.

## Practice

## Quiz

## Daily challenge

## Before you go

Find an object in your own code with a `status` or `mode` attribute and list its states and events as a table. Is every event in every state handled? Tomorrow: the Command pattern, which turns actions into objects you can store, repeat and undo.